Authors: Matt Kuehr, Matthew Sawyer

Emails: mck0063@auburn.edu, mss0096@auburn.edu

# Re-training the Best Model: LSTM with WordLevel tokenizer

In [1]:
import os
import sys
sys.path.append('..')
import torch
import torch.nn as nn
import torch.optim as optim
from scripts.preprocess import get_data
from scripts.models import SentimentRNN
import time

c:\Users\Matt\Desktop\old_repo\COMP-6630-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TOKENIZER_TYPE = "word"
MODEL_TYPE = "lstm"
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_DIR = "../checkpoints"

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
train_loader, val_loader, test_loader, vocab_size = get_data(tokenizer_type=TOKENIZER_TYPE, batch_size=BATCH_SIZE)
print(f"Vocab Size: {vocab_size}")

Repo card metadata block was not found. Setting CardData to empty.


Vocab Size: 26486


In [4]:
EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
EPOCHS = 25
LEARNING_RATE = 1e-3

model = SentimentRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, 
                     N_LAYERS, BIDIRECTIONAL, DROPOUT, model_type=MODEL_TYPE)

model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss().to(DEVICE)

print(model)

SentimentRNN(
  (embedding): Embedding(26486, 64, padding_idx=0)
  (rnn): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (sig): Sigmoid()
)


In [5]:
def calculate_accuracy(preds, y):
    """
    Calculates accuracy per batch.

    Args:
        preds (torch.Tensor): The model predictions.
        y (torch.Tensor): The ground truth labels.

    Returns:
        torch.Tensor: The accuracy score.
    """
    rounded_preds = torch.round(preds)
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

def train(model, iterator, optimizer, criterion, device):
    """
    Trains the model for one epoch.

    Args:
        model (nn.Module): The model to be trained.
        iterator (DataLoader): The DataLoader for training data.
        optimizer (torch.optim.Optimizer): The optimizer for weight updates.
        criterion (nn.Module): The loss function.
        device (torch.device): The device to run training on.

    Returns:
        tuple: A tuple containing (average_epoch_loss, average_epoch_accuracy).
    """
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    
    for batch in iterator:
        optimizer.zero_grad()
        text = batch["input_ids"].to(device)
        labels = batch["label"].unsqueeze(1).to(device)
        lengths = batch["lengths"]
        
        predictions = model(text, lengths)
        loss = criterion(predictions, labels)
        acc = calculate_accuracy(predictions, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion, device):
    """
    Evaluates the model on a given dataset.

    Args:
        model (nn.Module): The model to be evaluated.
        iterator (DataLoader): The DataLoader for evaluation data.
        criterion (nn.Module): The loss function.
        device (torch.device): The device to run evaluation on.

    Returns:
        tuple: A tuple containing (average_epoch_loss, average_epoch_accuracy).
    """
    epoch_loss = 0
    epoch_acc = 0
    model.eval()
    
    with torch.no_grad():
        for batch in iterator:
            text = batch["input_ids"].to(device)
            labels = batch["label"].unsqueeze(1).to(device)
            lengths = batch["lengths"]
            
            predictions = model(text, lengths)
            loss = criterion(predictions, labels)
            acc = calculate_accuracy(predictions, labels)
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
best_valid_loss = float('inf')

print(f"Starting training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, DEVICE)
    valid_loss, valid_acc = evaluate(model, val_loader, criterion, DEVICE)
    
    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
    
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"model_epoch_{epoch+1}.pt")
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'valid_loss': valid_loss,
    }, checkpoint_path)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        print(f"\tNew best validation loss! Saved best_model.pt")

    print(f'Epoch: {epoch+1:02} | Epoch Time: {int(epoch_mins)}m {int(epoch_secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')
    print(f'\tCheckpoint saved to {checkpoint_path}')

Starting training for 25 epochs...
	New best validation loss! Saved best_model.pt
Epoch: 01 | Epoch Time: 0m 9s
	Train Loss: 0.561 | Train Acc: 69.92%
	 Val. Loss: 0.425 |  Val. Acc: 80.63%
	Checkpoint saved to ../checkpoints\model_epoch_1.pt
	New best validation loss! Saved best_model.pt
Epoch: 02 | Epoch Time: 0m 8s
	Train Loss: 0.461 | Train Acc: 77.88%
	 Val. Loss: 0.352 |  Val. Acc: 84.23%
	Checkpoint saved to ../checkpoints\model_epoch_2.pt
	New best validation loss! Saved best_model.pt
Epoch: 03 | Epoch Time: 0m 9s
	Train Loss: 0.412 | Train Acc: 81.01%
	 Val. Loss: 0.304 |  Val. Acc: 87.25%
	Checkpoint saved to ../checkpoints\model_epoch_3.pt
	New best validation loss! Saved best_model.pt
Epoch: 04 | Epoch Time: 0m 9s
	Train Loss: 0.376 | Train Acc: 82.96%
	 Val. Loss: 0.267 |  Val. Acc: 89.17%
	Checkpoint saved to ../checkpoints\model_epoch_4.pt
	New best validation loss! Saved best_model.pt
Epoch: 05 | Epoch Time: 0m 9s
	Train Loss: 0.343 | Train Acc: 84.86%
	 Val. Loss: 0.23

In [7]:
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt')))
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

print(f'Final Test Acc: {test_acc*100:.2f}%')

Final Test Acc: 99.39%
